# 01 - Data Understanding

Notebook này dùng để load dữ liệu Telco Customer Churn, kiểm tra chất lượng dữ liệu, xử lý kiểu dữ liệu cơ bản và xuất `features.csv`, `target.csv` cho các bước sau.

In [1]:
from pathlib import Path
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'base.yaml'

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

raw_path = PROJECT_ROOT / config['paths']['raw_data']
processed_dir = PROJECT_ROOT / config['paths']['processed_dir']
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)
df.head()

ModuleNotFoundError: No module named 'pandas'

## Kiểm tra ban đầu

Kiểm tra kích thước dữ liệu, schema, missing values, duplicate và số lượng giá trị duy nhất để hiểu cấu trúc dữ liệu trước khi xử lý.

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False))
print('Duplicates:', df.duplicated().sum())
display(df.nunique().sort_values())

## Xử lý kiểu dữ liệu

`TotalCharges` thường được đọc dưới dạng object do có giá trị rỗng. Ta chuyển sang numeric bằng `pd.to_numeric(errors='coerce')` rồi xử lý các dòng bị thiếu sau chuyển đổi.

In [ ]:
df = df.copy()

if 'SeniorCitizen' in df.columns:
    df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'}).fillna(df['SeniorCitizen'])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_total_charges = df['TotalCharges'].isna().sum()
print('Missing TotalCharges after conversion:', missing_total_charges)

display(df.loc[df['TotalCharges'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(20))

Các dòng thiếu `TotalCharges` thường có `tenure = 0`, nghĩa là khách hàng mới chưa phát sinh tổng phí. Trong pipeline nền, ta điền `TotalCharges = 0` để giữ lại các quan sát này thay vì loại bỏ.

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

target_column = config['dataset']['target_column']
id_column = config['dataset']['id_column']

features = df.drop(columns=[target_column])
target = df[[id_column, target_column]] if id_column in df.columns else df[[target_column]]

features.to_csv(PROJECT_ROOT / config['paths']['features'], index=False)
target.to_csv(PROJECT_ROOT / config['paths']['target'], index=False)

print('Saved:', PROJECT_ROOT / config['paths']['features'])
print('Saved:', PROJECT_ROOT / config['paths']['target'])